# 03 · Exploratory analysis: intensity, how the wars are fought, displacement, media, and India
Each section ends with **So what** – the actionable implication. Points that rely on knowledge outside this
dataset are labelled **[Domain]**; everything else is computed here.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from src import config as C, viz

In [2]:
from src import insights
from src.cleaning import load
e = load("events"); s = load("sources")
cm = pd.read_parquet(C.PROCESSED / "case_month.parquet")
I = insights.run()
I["case_overview"].set_index("conflict_case").T

conflict_case,Russia–Ukraine,Israel–Hamas,Afghanistan–Pakistan,US–Iran
events,44555,9620,5985,895
deaths_best,447452,67933,31082,7929
deaths_low,270529,64292,28784,6878
deaths_high,782980,71893,39746,8742
peak_month,Mar 2022,Oct 2023,Feb 2017,Mar 2026
peak_month_deaths,27245,11211,762,2825
deaths_2026_ytd,64453,1191,2767,3965
deaths_2025,94764,14547,2649,2167
civilian_share,0.070,0.490,0.199,0.365
deaths_per_event,10.043,7.062,5.193,8.859


## 1. Intensity and tempo

In [3]:
viz.fig_monthly_deaths(cm)

In [4]:
viz.fig_case_small_multiples(cm)

In [5]:
viz.fig_violence_mix(I['violence_mix'])

**Reading.** Russia–Ukraine dominates absolute deaths (≈447k recorded, 2014–Sep 2026) and has held a
5–11k/month plateau since 2023. Israel–Hamas peaked in Oct 2023 and has fallen to ~130/month during 2026.
Afghanistan–Pakistan is lower-intensity but **rising since 2021**. US–Iran went from proxy and coalition strikes
to a direct state-on-state war in Feb–Mar 2026 (peak month of the case).

**So what.** Threat is not one number: RU–UA is a *sustained attrition* problem, US–Iran a *spike* problem,
Af–Pak a *creeping* problem. Each needs a different warning indicator (level, shock, trend).

## 2. How the wars are being fought (military lens)

### 2a. Attrition by side (focal dyads)

In [6]:
att = I["attrition"]
display(att[att.year >= 2022].round(2))
viz.fig_attrition(att)

,conflict_case,side_a_name,side_b_name,year,deaths_side_a,deaths_side_b,deaths_civilians,deaths_unknown_status,ratio_a_to_b
4,Afghanistan–Pakistan,Government of Afghanistan,Government of Pakistan,2022,3,2,11,0,1.500
5,Afghanistan–Pakistan,Government of Afghanistan,Government of Pakistan,2023,4,7,6,0,0.570
6,Afghanistan–Pakistan,Government of Afghanistan,Government of Pakistan,2024,14,8,21,18,1.750
7,Afghanistan–Pakistan,Government of Afghanistan,Government of Pakistan,2025,42,38,56,42,1.110
8,Afghanistan–Pakistan,Government of Afghanistan,Government of Pakistan,2026,84,104,463,34,0.810
17,Israel–Hamas,Government of Israel,Hamas,2022,3,4,4,1,0.750
18,Israel–Hamas,Government of Israel,Hamas,2023,567,181,17151,8456,3.130
19,Israel–Hamas,Government of Israel,Hamas,2024,230,113,9740,10878,2.040
20,Israel–Hamas,Government of Israel,Hamas,2025,85,299,3217,10526,0.280
21,Israel–Hamas,Government of Israel,Hamas,2026,4,320,227,502,0.010


**Reading.** UCDP *records* Russian fatalities outnumbering Ukrainian ones by a widening margin
(about 2:1 in 2022–23 to about 10:1 in 2026). **Caveat:** these are recorded, source-dependent figures. Ukrainian
memorial and OSINT databases document Russian losses more systematically than the reverse, so the ratio is an
upper bound of reporting, not a verified exchange ratio.

**So what [Domain].** A defender with ISR-cued precision fires and drones can impose disproportionate attrition.
That lesson matters for India's force design on both fronts.

### 2b. Front-line geography: the front widens rather than moves

In [7]:
fg = I["front_geography"]
display(fg.pivot(index="area", columns="year", values="share").mul(100).round(1).fillna(0))
viz.fig_front_geography(fg)

year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
area,,,,,,,,,,,,,
Belgorod oblast,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.400,0.400,1.500,2.000,4.900
Dnipropetrovsk oblast,0.100,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.500,0.900,1.000,5.500,7.100
Donetsk oblast,68.600,79.400,79.500,79.500,70.800,80.500,56.600,70.600,54.900,52.700,51.300,43.600,29.800
Kharkiv oblast,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,10.100,7.900,14.600,12.400,9.300
Kherson oblast,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,6.800,12.600,7.500,9.000,11.400
Kursk oblast,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.100,4.000,4.200,1.800
Kyiv oblast,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3.500,0.200,0.100,0.200,1.200
Luhansk oblast,31.000,20.600,20.300,20.500,29.200,19.500,42.500,29.400,8.800,9.000,7.700,3.900,3.900
Other areas,0.300,0.000,0.200,0.000,0.000,0.000,0.900,0.000,8.200,2.300,2.100,2.600,10.100


**Reading.** Donetsk's share of geolocated events falls from about 55% (2022) to 30% (2026), while
Zaporizhzhia, Dnipropetrovsk, Sumy and Russia's Belgorod and Kursk grow. Violence is spreading **along and across**
the line rather than moving with a breakthrough.

**So what.** Static, wide fronts favour long-range strike, counter-drone and logistics depth. Warning systems
should track the *spread* of activity (number of active areas, concentration), which we feed into the models.

In [8]:
viz.fig_theatre_spread(I['theatre_spread'])

In [9]:
viz.fig_country_map(I['country_deaths'])

### 2c. Afghanistan–Pakistan: a second, state-on-state front opens

In [10]:
display(I["afpak_provinces"].pivot_table(index=["country_iso3", "admin1"], columns="year", values="deaths",
                                           aggfunc="sum").fillna(0).astype(int))
viz.fig_afpak(I["afpak_provinces"])

year                                        2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  2026
country_iso3 admin1                                                                                                     
AFG          (country-level / unspecified)     0    18    80   107    48     0     0     0     1     3     3    48    99
             Badakhshan province               0     0     0     0     0     3     1     0    15    25    10    17     4
             Badghis province                  0     0     0     0     0    11     0     0     4     0     0     2     0
             Baghlan province                  0     1     0     0     1     1     1    15    34    33    12     3     0
             Balkh province                    0     0    14     0    22     0     0     0    94    31     2     0     0
             Bamyan province                   0     0     0     0     0     0    19     0     0     0     7     0     0
             Daykundi province                 0     0     0     0     0     3     0     0     1     0    14     0     0
             Farah province                    1    54     5     1     5     0     0     1     4     0     0     0     0
             Faryab province                   0     0     5     0   145     1     0     1     4     0     7     2     0
             Ghazni province                   0     2    10     0    16     4     0     1     3     4     1     0     0
             Ghor province                     0     0    30    21     0     0     0     7     9     1    13     1     1
             Hilmand province                  0    21     3     2     6     0     0     0     4     0     0    25    10
             Hirat province                    0     0     0    37     3     1     3    22    83    25     9     0    17
             Jawzjan province                  0     0     9   260   557     0     0     0     0     1     0     0     1
             Kabul province                    0     0   144   292   476   159   237   227   323    94    41    18   284
             Kandahar province                 0     0     0     8     2     1     1    69    12     0    33    70    39
             Kapisa province                   0     0     0     2     2     0     6     1     2     0     0     1     0
             Khost province                    1     6    10     5    65     1     5     0    43     0    15    16    56
             Kunar province                    0     0    17   121   333   209    94    19    27    30    12     7   110
             Kunduz province                   0     4     5     4     0     3     0    84    84     1     2    20     0
             Laghman province                  0     0     0     0    16     1     0     0     8     1     0     1     0
             Logar province                    0     1     2     3     4     0    18     1     4     0     3     1     0
             Nangarhar province                0   769  2190  2409  1999   926    94   171    54    15    16    20    43
             Nimroz province                   0     0     0     0     1     0     0     0     0     9     4     0     1
             Nuristan province                 0     0     0     4    87     0     0     0     0     1     0     0     3
             Paktika province                  0     2    11     0     3     0     1     0     2    10    64    33    33
             Paktya province                   0     0     0     4    40    11     0     2     3     0    30     0    46
             Panjsher province                 0     0     0     0     0     0     0    44    34     0     1     0     0
             Parwan province                   0     0     0     0     0     0     6    20     2     4     5     0     0
             Samangan province                 0     1     0     0     1     0     0     0    31     0     0     0     0
             Sari Pul province                 0     0     0    47    25     0     0     0    34     0     4     0     0
       

In [11]:
viz.fig_afpak_interstate(e)

**Reading.** TTP-led violence has concentrated in **Khyber Pakhtunkhwa** (FATA merged), rising every year
since 2021. Since Oct 2025 the **Afghan and Pakistani governments have fought each other directly**: 131 deaths in
Oct 2025 and a peak of 411 in Mar 2026.

**So what (India).** Pakistan now faces active hostilities on its western border. **[Domain]** That stretches its
army and may reduce its capacity on the eastern front, or raise the incentive for diversionary activity along
the LoC. Either way, India needs closer watch on the western front, not less.

### 2d. Campaign seasons

In [12]:
viz.fig_seasonality(I['seasonality'])

**Reading.** RU–UA deaths run about 15–35% above trend in **May–August** and about 25–35% below in **Dec–Jan**
(the summer campaigning season versus winter and the *rasputitsa*). Af–Pak peaks in **Sep–Oct**.

**So what.** Seasonal priors are a cheap, explainable addition to warning; our models use them.

## 3. Media attention vs deaths (original index)

In [13]:
md_, st = I["media"], I["stats"]["media"]
print({k: round(float(v), 3) for k, v in st.items()})
viz.fig_media_scatter(md_)

{'elasticity_deaths': 0.009, 'elasticity_events': 0.959, 'r2': 0.964, 'n': 510.0, 'lead_coef': -0.047, 'lead_p': 0.502}


In [14]:
viz.fig_media_gap(md_)

In [15]:
I['indian_press']

,conflict_case,refs,indian_refs,events,events_with_indian_source,indian_ref_share,events_covered_share
0,Afghanistan–Pakistan,10733,1137,5985,"1,025.000",0.106,0.171
1,Israel–Hamas,15827,1,9620,1.000,0.000,0.000
2,Russia–Ukraine,56884,1,44555,1.000,0.000,0.000
3,US–Iran,1962,0,895,0.000,0.000,0.000


**Reading.** Once the number of incidents is known, **deaths add essentially nothing** to how many references
a month receives (elasticity ≈ 0.01 vs ≈ 0.96 for events). One mass-casualty strike counts as one story.
The attention gap does **not** lead next month's escalation (p ≈ 0.5), so media volume is not a warning signal.
Indian outlets (PTI) appear in **17% of Af–Pak events** and almost none of the other three theatres.

**So what (India).** India's open-source picture of West Asia and Europe depends almost entirely on foreign
reporting. Building indigenous OSINT coverage of the Gulf and Iran is a cheap capability gain.

## 4. Displacement and humanitarian planning

In [16]:
viz.fig_displacement(I['displacement'])

In [17]:
display(I["demographics_profile"].sort_values("year").groupby("origin_iso3").tail(1).round(3))
viz.fig_demographics(I["demographics_profile"])

,origin_iso3,year,people_specified,women_share,children_share,under5_share,elderly_share,women_children_share
23,IRN,2025,132585,0.430,0.226,0.025,0.131,0.546
11,AFG,2025,1288989,0.430,0.384,0.101,0.051,0.630
38,PAK,2025,85926,0.268,0.165,0.025,0.064,0.355
26,ISR,2025,1092,0.402,0.278,0.061,0.060,0.555
62,RUS,2025,65807,0.459,0.272,0.054,0.070,0.601
50,PSE,2025,33173,0.241,0.162,0.041,0.034,0.329
74,UKR,2025,3513034,0.582,0.292,0.047,0.094,0.721


In [18]:
lk, st2 = I["displacement_link"], I["stats"]["displacement_link"]
print({k: round(float(v), 3) for k, v in st2.items()})

{'coef': 0.772, 'p': 0.178, 'spearman': 0.1, 'n': 77.0}


**Reading.** Ukraine: about 5.2M refugees and 3.7M IDPs at end-2025; **72%** of Ukrainian refugees with recorded
sex and age are women or children. Afghanistan: 63% women and children, 10% under five. At country-year level,
deaths explain little of the *change* in displaced stocks (Spearman ≈ 0.1): displacement responds to the
onset of war and to access, not to the monthly death count.

**So what.** HADR planning should key on **war onset and spread** (our early-warning flags), and stock
paediatric and maternal supplies. Displacement cannot be attributed to single events (README caveat).

## 5. India impact (data-derived parts)

In [19]:
ie = I["india_exposure"]
display(ie[ie.year >= 2024].pivot_table(index="india_tag", columns="year", values=["events", "deaths"], aggfunc="sum").fillna(0))
viz.fig_india_exposure(ie)

deaths                 events              
year                                     2024    2025    2026   2024   2025   2026
india_tag                                                                         
Gulf state (Indian diaspora / energy)   0.000   6.000  37.000  0.000  1.000 25.000
Indian Ocean (Sri Lanka)                0.000   0.000 104.000  0.000  0.000  1.000
Iran: Hormuz / Chabahar coast           0.000   0.000 299.000  0.000  0.000 61.000
Pakistan area adjoining India/LoC      35.000  52.000  30.000 19.000 18.000 11.000
Yemen (Red Sea / Bab el-Mandeb)       155.000 945.000   4.000 39.000 81.000  1.000

In [20]:
display(I["india_hosted"].pivot_table(index="origin_name", columns="year", values="total_international"))
viz.fig_india_hosted(I["india_hosted"])

year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
origin_name,,,,,,,,,,,,
Afghanistan,"11,700.000","12,924.000","11,948.000","13,501.000","14,848.000","15,689.000","15,132.000","16,247.000","15,039.000","12,224.000","9,764.000","8,377.000"
Iran (Islamic Rep. of),114.000,77.000,77.000,69.000,85.000,105.000,106.000,102.000,160.000,196.000,133.000,110.000
Palestinian,78.000,87.000,86.000,75.000,78.000,79.000,66.000,63.000,62.000,52.000,52.000,62.000
Russian Federation,NaN,5.000,10.000,7.000,7.000,14.000,12.000,15.000,34.000,62.000,145.000,329.000
Ukraine,5.000,13.000,8.000,6.000,7.000,9.000,5.000,5.000,58.000,66.000,89.000,98.000


In [21]:
ic = I["india_context"]
ic.pivot(index="year", columns="iso3", values="consumer_inflation_annual_pct").round(1)

iso3,AFG,IND,IRN,ISR,PAK,RUS,UKR
year,,,,,,,
2014,4.700,6.700,16.600,0.500,7.200,7.800,12.100
2015,-0.700,4.900,12.500,-0.600,2.500,15.500,48.700
2016,4.400,4.900,7.200,-0.500,3.800,7.000,13.900
2017,5.000,3.300,8.000,0.200,4.100,3.700,14.400
2018,0.600,3.900,18.000,0.800,5.100,2.900,11.000
2019,2.300,3.700,39.900,0.800,10.600,4.500,7.900
2020,5.600,6.600,30.600,-0.600,9.700,3.400,2.700
2021,5.100,5.100,43.400,1.500,9.500,6.700,9.400
2022,13.700,6.700,43.500,4.400,19.900,13.700,20.200


**Data-derived**
* In 2026 the US–Iran war hit **six Gulf states** (25 events, 37 deaths), **Iran's Hormuz/Chabahar coast** (61 events,
  299 deaths) and reached the **Indian Ocean**: a strike off Sri Lanka's west coast killed **104** on 4 Mar 2026.
* Yemen and the Red Sea approaches: 945 deaths in 2025.
* Pakistan areas adjoining India or the LoC (Punjab, Sindh, AJK, Gilgit-Baltistan): low but rising
  (2 deaths in 2022, 52 in 2025).
* India hosts about 8.4k Afghans (falling) and a fast-growing number of **Russian asylum seekers (5 → 329, 2015–25)**.

**[Domain] reasoning (not from this dataset)**
* About 9 million Indians live in the Gulf, so evacuation contingencies are needed (cf. Op Ganga 2022, Op Kaveri 2023, Op Ajay 2023).
* A large share of India's crude and LPG imports transit Hormuz. The Chabahar port and the INSTC corridor run
  through the 2026 strike zone.
* Russia remains India's largest legacy arms supplier and Israel a key supplier of UAVs, missiles and sensors.
  Both suppliers are at war, which risks spares and delivery delays.